# End-to-end baseline synthesis

**P0 Essential · D3 Synthesis · 150 minutes**

Deliver one traceable evidence chain from source identity to a leakage-safe baseline and bounded conclusion.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pathlib import Path

def locate(relative: str, local_name: str | None = None) -> Path:
    candidates = []
    if local_name:
        candidates.append(Path.cwd() / local_name)
    candidates.extend(root / relative for root in [Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Cannot find {relative}. Run from the course clone or place the downloaded data beside this notebook."
    )


In [ ]:
path = locate('datasets/teaching/predictive-maintenance/observations.csv', 'observations.csv')
data = pd.read_csv(path)
assert data['UDI'].notna().all() and data['UDI'].is_unique
assert set(data['Machine failure'].unique()) <= {0, 1}
data.shape

## Task

Implement `run_baseline`. Protect feature-time and split boundaries, compare with a dummy classifier on the same test cases, and return a machine-readable result record.

In [ ]:
def run_baseline(frame: pd.DataFrame) -> dict:
    target = 'Machine failure'
    leakage = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
    identifiers = ['UDI', 'Product ID']
    features = [column for column in frame.columns
                if column not in [target, *leakage, *identifiers]]
    X_train, X_test, y_train, y_test = train_test_split(
        frame[features], frame[target], test_size=0.25, random_state=20260718,
        stratify=frame[target])
    numeric = [column for column in features if column != 'Type']
    preprocessing = ColumnTransformer([
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')),
                              ('scale', StandardScaler())]), numeric),
        ('category', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                               ('encode', OneHotEncoder(handle_unknown='ignore'))]),
         ['Type']),
    ])
    model = Pipeline([('preprocess', preprocessing),
                      ('model', LogisticRegression(max_iter=1000,
                                                   class_weight='balanced'))])
    dummy = DummyClassifier(strategy='prior')
    model.fit(X_train, y_train)
    dummy.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    baseline_scores = dummy.predict_proba(X_test)[:, 1]
    prediction = scores >= 0.5
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    return {
        'data_id': 'uci-601-teaching-v1',
        'split_id': 'stratified-test-0.25-seed-20260718',
        'rows': int(len(frame)),
        'test_rows': int(len(y_test)),
        'prevalence': float(y_test.mean()),
        'dummy_pr_auc': float(average_precision_score(y_test, baseline_scores)),
        'model_pr_auc': float(average_precision_score(y_test, scores)),
        'confusion': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
        'excluded_target_relatives': leakage,
        'limitation': 'Synthetic row-level evidence; not validated for a real factory.',
    }

In [ ]:
result = run_baseline(data)
assert result['rows'] == 10_000
assert result['test_rows'] == 2_500
assert result['model_pr_auc'] > result['dummy_pr_auc']
assert sum(result['confusion'].values()) == result['test_rows']
assert set(result['excluded_target_relatives']) == {'TWF','HDF','PWF','OSF','RNF'}
assert 'real factory' in result['limitation']
result

## Delivery

Complete the public dataset-datasheet, result-card, data-contract, reproducibility-manifest, and AI-log templates. Add one toy-oracle test and one smoke command. Your conclusion must state the unit, synthetic population, primary metric, baseline comparison, error counts, and external-validity limit.

**Instructor note.** This is the clean-clone exit exercise. Review the evidence chain rather than rewarding metric optimisation. Introduce one feature-role or split defect during oral verification and ask the student to repair it.